In [1]:
# Install coxkan
! pip install coxkan

# Hyperparameter searching

How to perform hyperparameter searches (sweeps) using the `coxkan.hyperparameter_search` module

In [ ]:
from coxkan import CoxKAN
from coxkan.datasets import create_dataset
from coxkan.hyperparam_search import Sweep
from sklearn.model_selection import train_test_split
import numpy as np
import torch
import pandas as pd

from functools import partial

# 1) Permitir objetos que aparecen en los checkpoints (pandas + numpy)
torch.serialization.add_safe_globals([
    pd.core.indexes.base.Index,
    pd.core.indexes.base._new_Index,
    np.core.multiarray._reconstruct,
    np.ndarray,
    np.dtype,
])

# 2) Forzar que torch.load NO use weights_only=True (PyTorch>=2.6 lo puso por defecto)
_torch_load_orig = torch.load
def _torch_load_no_weights_only(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _torch_load_orig(*args, **kwargs)
torch.load = _torch_load_no_weights_only

# Generate a synthetic dataset
log_partial_hazard = lambda x1, x2: x1**2 + x2**2
df = create_dataset(log_partial_hazard, baseline_hazard=0.01, n_samples=1000)

df_train_full, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_train_full, test_size=0.2, random_state=42)

Concordance index of true expression: 0.6139


C:\Users\ernes\AppData\Local\Temp\ipykernel_15472\1825147850.py:12: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.multiarray.
  np.core.multiarray._reconstruct,


### Sweep using validation set

In [13]:


sweep = Sweep(search_config='./search_config.yml')

sweep.run_val(df_train, df_val, duration_col='duration', event_col='event', 
              n_trials=10, # number of trials
              n_runs_per_trial=1, # number of runs for each set of hyperparameters
              save_params='./result.yml',
              seed=42) # save the best hyperparameters to this file

[I 2025-10-27 14:57:22,029] A new study created in memory with name: no-name-3788341a-4e0b-4219-920b-f4b186f016d0
  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:05<?, ?it/s]

[W 2025-10-27 14:57:27,970] Trial 0 failed with parameters: {'num_hidden': 0, 'hidden_dim': 5, 'base_fun': 'silu', 'grid': 3, 'k': 3, 'noise_scale': 0.031198904067240532, 'noise_scale_base': 0.011616722433639893, 'early_stopping': True, 'lr': 0.013311216080736894, 'steps': 52, 'lamb': 0.014548647782429914, 'lamb_entropy': 13, 'lamb_coef': 1, 'prune_threshold': 0.009091248360355032} because of the following error: UnpicklingError("Weights only load failed. In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.\nPlease file an issue with the following so that we can make `weights_only=True` compatible with your use case: WeightsUnpickler error: Can only build Tensor, Parameter, OrderedDict or types allowlisted via `add_safe_globals`, but got <class '

UnpicklingError: Weights only load failed. In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
Please file an issue with the following so that we can make `weights_only=True` compatible with your use case: WeightsUnpickler error: Can only build Tensor, Parameter, OrderedDict or types allowlisted via `add_safe_globals`, but got <class 'numpy.dtypes.ObjectDType'>

Check the documentation of torch.load to learn more about types accepted by default with weights_only https://pytorch.org/docs/stable/generated/torch.load.html.

In [ ]:
import yaml 

with open('./result.yml', 'r') as f:
    config = yaml.safe_load(f)

ckan = CoxKAN(**config['init_params'])

if config['train_params']['early_stopping']:
    ckan.train(df_train, df_val, duration_col='duration', event_col='event', **config['train_params'])
else:
    ckan.train(df_train_full, None, duration_col='duration', event_col='event', **config['train_params'])

ckan.prune_edges(config['prune_threshold'])

ckan.cindex(df_test)

### Cross-Validation sweep

In [ ]:
study = sweep.run_cv(df_train, duration_col='duration', event_col='event', 
                     n_trials=10,
                     n_folds=4,
                     save_params='./result_cv.yml',
                     verbose=0)


In [ ]:
import yaml 

with open('./result_cv.yml', 'r') as f:
    config = yaml.safe_load(f)

ckan = CoxKAN(**config['init_params'])

if config['train_params']['early_stopping']:
    ckan.train(df_train, df_val, duration_col='duration', event_col='event', **config['train_params'])
else:
    ckan.train(df_train_full, None, duration_col='duration', event_col='event', **config['train_params'])

ckan.prune_edges(config['prune_threshold'])

ckan.cindex(df_test)